# 📘 Module 4.2 – Contour Properties
📌 Goal: Measure and analyze detected objects

🧠 ONE-LINE IDEA

Contour properties tell us how big, where, and what shape an object is

### 🔹 STEP 0 – BASE SETUP (SAME PIPELINE, VERY IMPORTANT)


In [3]:
import cv2
import numpy as np

# Load image
image = cv2.imread("image.jpg")
img = cv2.resize(image, (450, 300))
if img is None:
    print("Image not found")
    exit()

# Preprocessing
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
blur = cv2.GaussianBlur(gray, (5, 5), 0)

_, binary = cv2.threshold(
    blur, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU
)

# Find contours
contours, _ = cv2.findContours(
    binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
)

print("Total contours found:", len(contours))


Total contours found: 47


### ✅ 1️⃣ CONTOUR AREA

📌 Measures size of object
📌 Used to remove small noise

Example – Area of Each Contour

In [4]:
for i, cnt in enumerate(contours):
    area = cv2.contourArea(cnt)
    print(f"Contour {i} Area:", area)


Contour 0 Area: 0.0
Contour 1 Area: 34.5
Contour 2 Area: 24.0
Contour 3 Area: 16.5
Contour 4 Area: 97.5
Contour 5 Area: 0.0
Contour 6 Area: 1.0
Contour 7 Area: 364.0
Contour 8 Area: 0.0
Contour 9 Area: 2.5
Contour 10 Area: 1.0
Contour 11 Area: 12.0
Contour 12 Area: 0.0
Contour 13 Area: 81.0
Contour 14 Area: 2.0
Contour 15 Area: 96.0
Contour 16 Area: 0.5
Contour 17 Area: 0.5
Contour 18 Area: 1.5
Contour 19 Area: 512.5
Contour 20 Area: 405.5
Contour 21 Area: 0.0
Contour 22 Area: 2710.5
Contour 23 Area: 0.0
Contour 24 Area: 0.5
Contour 25 Area: 4.5
Contour 26 Area: 27.0
Contour 27 Area: 0.5
Contour 28 Area: 0.0
Contour 29 Area: 0.5
Contour 30 Area: 52.5
Contour 31 Area: 0.0
Contour 32 Area: 0.0
Contour 33 Area: 82.0
Contour 34 Area: 85.0
Contour 35 Area: 293.0
Contour 36 Area: 7.0
Contour 37 Area: 12.5
Contour 38 Area: 548.0
Contour 39 Area: 3.5
Contour 40 Area: 2.5
Contour 41 Area: 1.5
Contour 42 Area: 0.5
Contour 43 Area: 4.5
Contour 44 Area: 0.0
Contour 45 Area: 4.0
Contour 46 Area: 86

#### Ignore Small Objects (VERY IMPORTANT 🔥)

In [5]:
filtered_contours = []

for cnt in contours:
    if cv2.contourArea(cnt) > 500:   # threshold
        filtered_contours.append(cnt)

print("Contours after filtering:", len(filtered_contours))


Contours after filtering: 4


### ✅ 2️⃣ CONTOUR PERIMETER (ARC LENGTH)

📌 Measures boundary length

In [6]:
for i, cnt in enumerate(filtered_contours):
    perimeter = cv2.arcLength(cnt, True)
    print(f"Contour {i} Perimeter:", perimeter)


Contour 0 Perimeter: 173.09545266628265
Contour 1 Perimeter: 476.0315251350403
Contour 2 Perimeter: 144.7695518732071
Contour 3 Perimeter: 2835.4297025203705


### ✅ 3️⃣ CENTROID (CENTER OF OBJECT)

📌 Used in tracking, robotics, alignment

In [7]:
output = img.copy()

for cnt in filtered_contours:
    M = cv2.moments(cnt)

    if M["m00"] != 0:
        cx = int(M["m10"] / M["m00"])
        cy = int(M["m01"] / M["m00"])

        # Draw centroid
        cv2.circle(output, (cx, cy), 5, (255, 0, 0), -1)
        cv2.putText(
            output, "Center", (cx - 20, cy - 10),
            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 1
        )

cv2.imshow("Centroids", output)
cv2.waitKey(0)
cv2.destroyAllWindows()


### ✅ 4️⃣ BOUNDING BOX

- 📌 Smallest rectangle enclosing object
- 📌 Used in object detection

In [8]:
bbox_img = img.copy()

for cnt in filtered_contours:
    x, y, w, h = cv2.boundingRect(cnt)
    cv2.rectangle(bbox_img, (x, y), (x + w, y + h), (255, 0, 0), 2)

cv2.imshow("Bounding Boxes", bbox_img)
cv2.waitKey(0)
cv2.destroyAllWindows()


### ✅ 5️⃣ CONVEX HULL

- 📌 Tightest convex shape around object
- 📌 Used in shape analysis & defect detection

In [9]:
hull_img = img.copy()

for cnt in filtered_contours:
    hull = cv2.convexHull(cnt)
    cv2.drawContours(hull_img, [hull], -1, (0, 255, 0), 2)

cv2.imshow("Convex Hull", hull_img)
cv2.waitKey(0)
cv2.destroyAllWindows()


# 🔍 ALL PROPERTIES TOGETHER (IMPORTANT VIEW)

In [10]:
final = img.copy()

for cnt in filtered_contours:
    area = cv2.contourArea(cnt)
    perimeter = cv2.arcLength(cnt, True)

    x, y, w, h = cv2.boundingRect(cnt)
    cv2.rectangle(final, (x, y), (x + w, y + h), (255, 0, 0), 2)

    M = cv2.moments(cnt)
    if M["m00"] != 0:
        cx = int(M["m10"] / M["m00"])
        cy = int(M["m01"] / M["m00"])
        cv2.circle(final, (cx, cy), 4, (0, 0, 255), -1)

    cv2.putText(
        final,
        f"A:{int(area)} P:{int(perimeter)}",
        (x, y - 5),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.5,
        (0, 5, 0),
        1
    )

cv2.imshow("Contour Properties", final)
cv2.waitKey(0)
cv2.destroyAllWindows()


#### 🧠 SUMMARY TABLE (EXAM GOLD ⭐)
Property	Function
- Area	      ->cv2.contourArea()
- Perimeter	      ->cv2.arcLength()
- Centroid	      ->cv2.moments()
- Bounding box	      ->cv2.boundingRect()
- Convex hull	      ->cv2.convexHull()